# Week5_ex2 — Three-phase faults on the loaded 37-bus system

Reproduces Problem 8.28 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.): a bolted three-phase fault at each of the 37 buses of the Example 6.13 system, with the pre-fault profile taken from the solved power flow and checked against PowerWorld's fault analysis.

![PowerWorld one-line](Images/Week5_ex2_result_image.png)

In [1]:
import warnings
import numpy as np
import pandas as pd
from case37 import S_BASE, BUSES, build_case, solve_pw, residual_band, compare, summarize
from fault37 import GEN_Z, pw_fault_table, fault_results, fault_measure

# 1. FutureWarning from pandas inside pandapower's branch builder; it prints the local install path
warnings.filterwarnings("ignore", category=FutureWarning)

# 2. Generator positive-sequence impedances added to the Ybus diagonal (pu on S_BASE)
pd.DataFrame([(b, r, x) for b, (r, x) in GEN_Z.items()],
             columns=["Bus", "R [pu]", "X'' [pu]"]).set_index("Bus")

,R [pu],X'' [pu]
Bus,,
28,0.00000,0.050
31,0.00000,0.050
44,0.00400,0.127
50,0.00350,0.150
53,0.00132,0.050
54,0.00112,0.088


In [2]:
# 3. Pre-fault state = the Example 6.13 power flow, solved with frozen controls and Mvar back-off
net = solve_pw(build_case(), numba=False)
pre = pd.DataFrame({"Voltage [pu]": net.res_bus.vm_pu, "Angle [deg]": net.res_bus.va_degree})
pre.describe().loc[["min", "max"]].round(5)

,Voltage [pu],Angle [deg]
min,0.99019,-24.75716
max,1.03000,-12.79114


In [3]:
# 4. Residual band: how far each fault quantity moves when the pre-fault solution carries
#    PowerWorld's 0.1 MVA mismatch at any bus
band = residual_band(build_case, fault_measure)

# 5. Tolerance = half of the printed digit (5 decimals) + residual band
pw = pw_fault_table()
pp_res = fault_results(net)
SRC = "PowerWorld Fault Definitions (Run Mode)"
rows = []
for b in pw.index:
    for col, group in [("R_th [pu]", "Thevenin R"), ("X_th [pu]", "Thevenin X"),
                       ("I [pu]", "Fault current"), ("Angle [deg]", "Fault angle")]:
        rows.append((group, f"{group} bus {b}", pp_res.at[b, col], pw.at[b, col],
                     5e-6 + band[f"{col} {b}"], SRC))
table = compare(rows)
summarize(table)

,Group,Source,Rows,Max abs diff,Tightest tolerance,Smallest margin,All pass
0,Thevenin R,PowerWorld Fault Definitions (Run Mode),37,0.000005,0.000007,0.000003,True
1,Thevenin X,PowerWorld Fault Definitions (Run Mode),37,0.000005,0.000007,0.000003,True
2,Fault current,PowerWorld Fault Definitions (Run Mode),37,0.000261,0.000772,0.000758,True
3,Fault angle,PowerWorld Fault Definitions (Run Mode),37,0.000150,0.005952,0.005926,True


In [4]:
# 6. Fault level at every bus; kA uses each bus's own base current S_BASE / (sqrt(3) * kV)
kv = pd.Series({b: v for b, _, v in BUSES})
out = pp_res.copy()
out.insert(0, "Name", [n for _, n, _ in BUSES])
out.insert(1, "kV", kv)
out["I [kA]"] = out["I [pu]"] * S_BASE / (np.sqrt(3) * kv)
out["S_sc [MVA]"] = out["I [pu]"] * pre["Voltage [pu]"] * S_BASE
out.sort_values("I [pu]", ascending=False).round(4)

,Name,kV,I [pu],Angle [deg],R_th [pu],X_th [pu],I [kA],S_sc [MVA]
bus,,,,,,,,
28,ELM345,345.0,64.0245,-100.6382,0.0006,0.0161,10.7144,6594.5280
31,SLACK345,345.0,54.3463,-101.0074,0.0012,0.0189,9.0947,5597.6710
53,PEAR138,138.0,43.1929,-103.4978,0.0028,0.0230,18.0706,4325.5865
38,PINE345,345.0,40.9909,-101.1526,0.0020,0.0249,6.8597,4200.3791
47,CEDAR138,138.0,40.3980,-102.1597,0.0036,0.0244,16.9013,4032.4715
54,PEAR69,69.0,39.0181,-101.0201,0.0051,0.0254,32.6480,3940.8288
39,PINE138,138.0,37.8774,-100.7802,0.0034,0.0266,15.8468,3842.2558
15,PECAN69,69.0,37.6009,-97.9906,0.0067,0.0259,31.4621,3783.1883
29,ELM138,138.0,35.7017,-100.9689,0.0022,0.0286,14.9365,3657.5092
